# TCI, TCC, and observational coupling plots

This notebook consolidates model TCI difference maps, model TCC difference maps, observational TCI maps, and regional time series. Select one product in the user setup cell.


In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import os
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")
WORKFLOW_NAME = "analysis_lac"

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = WORK_DIR
DIAGNOSTIC_DIR = PROJECT_ROOT / "diagnostic"
if not (DIAGNOSTIC_DIR / "util").is_dir():
    raise FileNotFoundError(
        f"Invalid project root: {PROJECT_ROOT}. Set WORK_DIR in the top-level parameter cell."
    )
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs


In [ ]:
# === Notebook path configuration ===
from pathlib import Path
import sys

CWD = WORK_DIR
DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"

if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import ttest_ind
from matplotlib import cm
from matplotlib import ticker
from scipy.ndimage import gaussian_filter
from configs.land_atmosphere_experiment_configs import get_experiment_dict
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm
import string
import cmocean
from util.dask_helpers import (
    configure_dask,
    merge_chunks,
    open_dataset_lazy,
    open_mfdataset_lazy,
    persist_if_dask,
)


In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import string
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import BoundaryNorm

class TCIPlotter:
    def __init__(self, tci_files_dict, windows, landmask_file=None, compute_ensemble_mean=False, dask_chunks=None, persist_inputs=False):
        self.tci_files = tci_files_dict
        self.windows = list(windows.keys())
        self.compute_ensemble_mean = compute_ensemble_mean
        self.dask_chunks = merge_chunks(dask_chunks)
        self.persist_inputs = persist_inputs
        self.projection = ccrs.Robinson()

        if landmask_file and os.path.exists(landmask_file):
            ds = open_dataset_lazy(landmask_file, chunks=self.dask_chunks)
            landmask = ds['landfrac']
            if 'lon' in landmask.coords and landmask.lon.max() > 180:
                landmask['lon'] = ((landmask['lon'] + 180) % 360) - 180
                landmask = landmask.sortby('lon')
            self.landmask = landmask  
        else:
            self.landmask = None

    def _load_ensemble_data(self, exp, window, region=None):
        meta = self.tci_files[exp]
        nens = meta["nens"]
        ftemplate = meta["template"]
        fpath = meta["path"] % {"exp": exp}

        files = []
        for ens in range(1, nens + 1):
            fname = ftemplate % {"window": window, "ensemble": f"{ens:02d}", "exp": exp}
            full_path = os.path.join(fpath, fname)
            if os.path.exists(full_path):
                files.append(full_path)

        if not files:
            raise FileNotFoundError(f"[ERROR] No TCI files found for {exp} {window}")

        ds = open_mfdataset_lazy(
            files,
            combine='nested',
            concat_dim='ensemble',
            coords='minimal',
            compat='override',
            chunks=self.dask_chunks,
        )
        da = ds['TCI'].assign_coords(ensemble=np.arange(len(files)))
        da = persist_if_dask(da, self.persist_inputs)
        
        if region:
            da = da.sel(lat=slice(*region['lat']), lon=slice(*region['lon']))
        
        return da

    def _compute_dart_minus_reference_maps(self, dart_exps, reference_exp, percentile,
                                           region=None, significance=False,
                                           n_boot=1000, alpha=0.05):
        diff_maps = {}
        sig_masks = {}
        global_vmin = np.inf
        global_vmax = -np.inf

        for dart_exp in dart_exps:
            diff_maps[dart_exp] = {}
            sig_masks[dart_exp] = {}

            for window in self.windows:
                try:
                    dart = self._load_ensemble_data(dart_exp, window, region)
                    ref = self._load_ensemble_data(reference_exp, window, region)
                except Exception as e:
                    print(f"[ERROR] Failed to load data for {dart_exp} vs {reference_exp} in {window}: {e}")
                    continue

                dim_list = []
                if 'ensemble' in dart.dims:
                    dim_list.append('ensemble')
                if 'time' in dart.dims and dart.sizes.get('time', 1) > 1:
                    dim_list.append('time')
                    
                dart = dart.where(np.isfinite(dart))
                ref = ref.where(np.isfinite(ref))
                
                # Compute quantiles
                dart_p = dart.quantile(percentile, dim=dim_list, skipna=True)
                ref_p = ref.quantile(percentile, dim=dim_list, skipna=True)

                if self.landmask is not None:
                    landmask_interp = self.landmask.interp_like(dart_p, method='nearest')
                    dart_p = dart_p.where(landmask_interp > 0.5)
                    ref_p = ref_p.where(landmask_interp > 0.5)

                # Compute difference
                diff = dart_p - ref_p
                diff_maps[dart_exp][window] = diff

                diff_min = diff.min(skipna=True).compute().item()
                diff_max = diff.max(skipna=True).compute().item()
                global_vmin = min(global_vmin, float(diff_min))
                global_vmax = max(global_vmax, float(diff_max))

                if significance:
                    # Compute difference for significance testing
                    diffs = persist_if_dask(dart - ref, True)
                    diffs = diffs.where(self.landmask > 0.5)  # Mask out ocean areas

                    if 'time' in diffs.dims and diffs.sizes['time'] > 1:
                        diffs = diffs.mean(dim='time')  # Average over time if available
                    
                    # Bootstrapping for significance calculation
                    boot_means = []
                    for _ in range(n_boot):
                        resample = diffs.isel(ensemble=np.random.randint(0, diffs.sizes['ensemble'], diffs.sizes['ensemble']))
                        boot_mean = resample.mean(dim='ensemble', skipna=True)
                        boot_means.append(boot_mean)
                    boot_stack = xr.concat(boot_means, dim='boot')
                    
                    # Calculate p-value based on bootstrapped values
                    pval = ((boot_stack < 0).sum(dim='boot') / n_boot).where(diff < 0,
                            (boot_stack > 0).sum(dim='boot') / n_boot)
                    
                    # Determine significance: p-value < alpha
                    sig_mask = pval < alpha
                    sig_masks[dart_exp][window] = sig_mask

        return diff_maps, sig_masks, global_vmin, global_vmax

    def _plot_difference_panels(self, diff_maps, sig_masks, dart_exps, reference_exp,
                                cmap, vmin, vmax, significance, levels=None,
                                savepath=None, dpi=150):
        nrows = len(dart_exps)
        ncols = len(self.windows)
        fig, axes = plt.subplots(
            nrows=nrows, ncols=ncols,
            figsize=(4.0 * ncols, 2.5 * nrows),
            subplot_kw={'projection': self.projection},
            dpi=dpi
        )

        if nrows == 1:
            axes = [axes]
        if ncols == 1:
            axes = [[ax] for ax in axes]

        bounds = np.linspace(vmin, vmax, 11) if levels is None else levels
        norm = BoundaryNorm(bounds, ncolors=256)
        cmap = cmap or plt.cm.RdBu_r

        row_labels = [f"({letter})" for letter in string.ascii_lowercase[:ncols]]
        subplot_labels = row_labels * 2  # Repeat for second row
        
        for i, dart_exp in enumerate(dart_exps):
            for j, window in enumerate(self.windows):
                ax = axes[i][j]
                idx = i * ncols + j
                diff = diff_maps[dart_exp].get(window)
                if diff is None:
                    ax.set_axis_off()
                    continue

                # Plot difference
                im = ax.contourf(diff['lon'], diff['lat'], diff.squeeze(),
                                 transform=ccrs.PlateCarree(),
                                 levels=bounds, cmap=cmap, norm=norm, extend='both')

                # Plot significance if available
                if significance and window in sig_masks[dart_exp]:
                    sig_mask = sig_masks[dart_exp][window].squeeze()
                    sig_mask_masked = xr.where(sig_mask, np.nan, 1).squeeze()

                    # Only plot if mask has valid (non-NaN) data
                    if not np.isnan(sig_mask_masked.values).all():
                        ax.contourf(sig_mask_masked['lon'], sig_mask_masked['lat'], sig_mask_masked,
                                    transform=ccrs.PlateCarree(), hatches=[".."], colors='none')

                ax.coastlines()
                ax.set_title(f"{subplot_labels[idx]} {dart_exp} - {reference_exp}\n{window}", fontsize=10)

        # Shared discrete colorbar
        cbar_ax1 = fig.add_axes([0.92, 0.60, 0.01, 0.30])
        sm1 = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm1.set_array([])
        fig.colorbar(sm1, cax=cbar_ax1, label="ΔTCI", extend='both')

        cbar_ax2 = fig.add_axes([0.92, 0.18, 0.01, 0.30])
        sm2 = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm2.set_array([])
        fig.colorbar(sm2, cax=cbar_ax2, label="ΔTCI", extend='both')

        plt.subplots_adjust(left=0.05, right=0.90, bottom=0.08, top=0.94, hspace=0.15, wspace=0.05)

        if savepath:
            plt.savefig(savepath, bbox_inches='tight')
            print(f"[SAVED] {savepath}")
        plt.show()

    def plot_dart_minus_reference_comparison(self, reference_exp='CAPTEN10',
                                              dart_exps=('DARTEN20', 'DARTEN40'),
                                              percentile=0.95, region=None,
                                              cmap=None, vmin=None, vmax=None,
                                              significance=False, n_boot=1000,
                                              alpha=0.05, levels=None,
                                              savepath=None, dpi=150):
        cmap = cmap or plt.cm.RdBu_r

        # Step 1: compute differences
        diff_maps, sig_masks, global_vmin, global_vmax = self._compute_dart_minus_reference_maps(
            dart_exps, reference_exp, percentile,
            region=region, significance=significance,
            n_boot=n_boot, alpha=alpha)

        # Step 2: determine color scale
        vmin = vmin if vmin is not None else global_vmin
        vmax = vmax if vmax is not None else global_vmax

        # Step 3: plot
        self._plot_difference_panels(diff_maps, sig_masks, dart_exps, reference_exp,
                                     cmap, vmin, vmax, significance,
                                     levels=levels, savepath=savepath, dpi=dpi)


In [ ]:
class TCCPlotter:
    def __init__(
            self, 
            tci_files_dict, 
            variable,
            windows, 
            landmask_file=None, 
            region=None,  
            compute_ensemble_mean=False
        ):
        self.tci_files = tci_files_dict
        self.variable = variable
        self.windows = list(windows.keys())
        self.projection = ccrs.Robinson()
        self.region = region 
        self.compute_ensemble_mean = compute_ensemble_mean
        
        if landmask_file and os.path.exists(landmask_file):
            ds = xr.open_dataset(landmask_file)
            landmask = ds['landfrac']
            if 'lon' in landmask.coords and landmask.lon.max() > 180:
                landmask['lon'] = ((landmask['lon'] + 180) % 360) - 180
                landmask = landmask.sortby('lon')
            if self.region:
                landmask = landmask.sel(
                    lat=slice(*self.region['lat']), 
                    lon=slice(*self.region['lon'])
                )
            self.landmask = landmask
        else: 
            self.landmask = None 

    def _load_ensemble_data(self, exp, window):
        meta = self.tci_files[exp]
        nens = meta["nens"]
        ftemplate = meta["template"]
        fpath = meta["path"] % {"exp": exp}

        files = []
        for ens in range(1, nens + 1):
            fname = ftemplate % {"exp": exp, "window": window, "ensemble": f"{ens:02d}"}
            full_path = os.path.join(fpath, fname)
            if os.path.exists(full_path):
                files.append(full_path)

        if not files:
            raise FileNotFoundError(f"[ERROR] No {self.variable} files found for {exp} {window}")

        da_list = []
        for k, f in enumerate(files):
            with xr.open_dataset(f) as ds:
                da_list.append(ds[self.variable].expand_dims(ensemble=[k]))

        da = xr.concat(da_list, dim='ensemble')
        if self.region:
            da = da.sel(lat=slice(*self.region['lat']), lon=slice(*self.region['lon']))
        return da

    def _compute_dart_minus_reference_maps(self, dart_exps, reference_exp, percentile,
                                           significance=False, n_boot=1000, alpha=0.05):
        diff_maps = {}
        sig_masks = {}
        global_vmin = np.inf
        global_vmax = -np.inf

        for dart_exp in dart_exps:
            diff_maps[dart_exp] = {}
            sig_masks[dart_exp] = {}
            for window in self.windows:
                try:
                    dart = self._load_ensemble_data(dart_exp, window)
                    ref = self._load_ensemble_data(reference_exp, window)
                except Exception as e:
                    print(f"[ERROR] Failed to load data for {dart_exp} vs {reference_exp} in {window}: {e}")
                    continue

                dim_list = []
                if 'ensemble' in dart.dims:
                    dim_list.append('ensemble')
                if 'time' in dart.dims and dart.sizes.get('time', 1) > 1:
                    dim_list.append('time')

                dart = dart.where(np.isfinite(dart))
                ref = ref.where(np.isfinite(ref))

                dart_p = dart.quantile(percentile, dim=dim_list, skipna=True)
                ref_p = ref.quantile(percentile, dim=dim_list, skipna=True)

                if not dart_p.lat.identical(ref_p.lat) or not dart_p.lon.identical(ref_p.lon):
                    ref_p = ref_p.interp_like(dart_p, method='nearest')

                if self.landmask is not None:
                    landmask_interp = self.landmask.interp_like(dart_p, method='nearest')
                    dart_p = dart_p.where(landmask_interp > 0.5)
                    ref_p = ref_p.where(landmask_interp > 0.5)

                diff = dart_p - ref_p
                diff_maps[dart_exp][window] = diff

                global_vmin = min(global_vmin, float(diff.min()))
                global_vmax = max(global_vmax, float(diff.max()))

                if significance:
                    diffs = dart - ref
                    if self.landmask is not None:
                        diffs = diffs.where(self.landmask > 0.5)

                    if 'time' in diffs.dims and diffs.sizes['time'] > 1:
                        diffs = diffs.mean(dim='time')

                    boot_means = []
                    for _ in range(n_boot):
                        resample = diffs.isel(ensemble=np.random.randint(0, diffs.sizes['ensemble'], diffs.sizes['ensemble']))
                        boot_mean = resample.mean(dim='ensemble', skipna=True)
                        boot_means.append(boot_mean)

                    boot_stack = xr.concat(boot_means, dim='boot')
                    pval = ((boot_stack < 0).sum(dim='boot') / n_boot).where(diff < 0,
                            (boot_stack > 0).sum(dim='boot') / n_boot)
                    sig_mask = pval < alpha
                    if self.landmask is not None:
                        sig_mask = sig_mask.where(self.landmask > 0.5)
                    sig_masks[dart_exp][window] = sig_mask

        return diff_maps, sig_masks, global_vmin, global_vmax

    def _plot_difference_panels(self, diff_maps, sig_masks, dart_exps, reference_exp,
                                cmap, vmin, vmax, significance, levels=None,
                                savepath=None, dpi=150):
        import matplotlib.ticker as mticker
        nrows = len(dart_exps)
        ncols = len(self.windows)
        fig, axes = plt.subplots(
            nrows=nrows, ncols=ncols,
            figsize=(4 * ncols, 3 * nrows),
            subplot_kw={'projection': self.projection},
            dpi=dpi)

        if nrows == 1:
            axes = [axes]
        if ncols == 1:
            axes = [[ax] for ax in axes]

        bounds = np.linspace(vmin, vmax, 11) if levels is None else levels
        norm = BoundaryNorm(bounds, ncolors=256)
        cmap = cmap or plt.cm.RdBu_r

        row_labels = [f"({letter})" for letter in string.ascii_lowercase[ncols:ncols*2]]
        subplot_labels = row_labels * 2

        for i, dart_exp in enumerate(dart_exps):
            for j, window in enumerate(self.windows):
                ax = axes[i][j]
                idx = i * ncols + j
                diff = diff_maps[dart_exp].get(window)
                if diff is None:
                    ax.set_axis_off()
                    continue

                im = ax.contourf(diff['lon'], diff['lat'], diff.squeeze(),
                                 transform=ccrs.PlateCarree(),
                                 levels=bounds, cmap=cmap, norm=norm, extend='both')

                if significance and window in sig_masks[dart_exp]:
                    sig_mask = sig_masks[dart_exp][window].squeeze()
                    sig_mask_masked = xr.where(sig_mask, np.nan, 1).squeeze()

                    if not np.isnan(sig_mask_masked.values).all():
                        ax.contourf(sig_mask_masked['lon'], sig_mask_masked['lat'], sig_mask_masked,
                                    transform=ccrs.PlateCarree(),
                                    hatches=[".."], colors='none')

                ax.coastlines()
                ax.set_title(f"{subplot_labels[idx]} {dart_exp} - {reference_exp}\n{window}", fontsize=10)

        # Colorbars
        cbar_ax1 = fig.add_axes([0.92, 0.60, 0.01, 0.30])
        sm1 = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm1.set_array([])
        cbar1 = fig.colorbar(sm1, cax=cbar_ax1, ticks=bounds, extend='both')
        cbar1.ax.tick_params(labelsize=10)
        cbar1.ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
        cbar1.set_label("ΔTCC", fontsize=10)

        cbar_ax2 = fig.add_axes([0.92, 0.18, 0.01, 0.30])
        sm2 = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm2.set_array([])
        cbar2 = fig.colorbar(sm2, cax=cbar_ax2, ticks=bounds, extend='both')
        cbar2.ax.tick_params(labelsize=10)
        cbar2.ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, prune='both'))
        cbar2.set_label("ΔTCC", fontsize=10)

        plt.subplots_adjust(left=0.05, right=0.90, bottom=0.08, top=0.94, hspace=0.15, wspace=0.05)

        if savepath:
            plt.savefig(savepath, bbox_inches='tight')
            print(f"[SAVED] {savepath}")
        plt.show()

    def plot_dart_minus_reference_comparison(
            self, reference_exp='CAPTEN10',
            dart_exps=('DARTEN20', 'DARTEN40'),
            percentile=0.95, cmap=None, 
            vmin=None, vmax=None,
            significance=False, n_boot=1000,
            alpha=0.05, levels=None,
            savepath=None, dpi=150):

        cmap = cmap or plt.cm.RdBu_r

        diff_maps, sig_masks, global_vmin, global_vmax = self._compute_dart_minus_reference_maps(
            dart_exps, reference_exp, percentile,
            significance=significance,
            n_boot=n_boot, alpha=alpha)

        vmin = vmin if vmin is not None else global_vmin
        vmax = vmax if vmax is not None else global_vmax

        self._plot_difference_panels(diff_maps, sig_masks, dart_exps, reference_exp,
                                     cmap, vmin, vmax, significance,
                                     levels=levels, savepath=savepath, dpi=dpi)


In [ ]:
# === Notebook path configuration ===
from pathlib import Path
import sys

CWD = WORK_DIR
DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"

if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs

import os
import re
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.dates import DateFormatter


class ObservedTCIPlotter:
    def __init__(self, tci_dir, regional_dir=None, fig_dir=None,
                 cmap='viridis', vmin=None, vmax=None):
        self.tci_dir = tci_dir
        self.regional_dir = regional_dir or tci_dir
        self.fig_dir = fig_dir or os.path.join(tci_dir, "figures")
        os.makedirs(self.fig_dir, exist_ok=True)

        self.cmap = cmap
        self.vmin = vmin
        self.vmax = vmax

    # ========== UTILS ==========

    def _extract_window_time(self, tag):
        match = re.search(r'\d{4}-\d{2}', tag)
        if match:
            return match.group(0)
        return tag

    
        # ========== MAP PLOTS ==========
    def plot_window_grid_map(self, sort_windows=True, projection="Robinson"):
        import matplotlib.gridspec as gridspec
    
        files = [
            f for f in os.listdir(self.tci_dir)
            if f.startswith("TCI_obs_window_") and f.endswith(".nc")
        ]
    
        if not files:
            print("[WARN] No window files found.")
            return
    
        # Sort by label like D01–15, D16–30, etc.
        if sort_windows:
            files.sort(key=lambda f: f.replace(".nc", "").split("_")[-1])
    
        nrows = len(files)
        fig = plt.figure(figsize=(12, 2.5 * nrows))
        gs = gridspec.GridSpec(nrows, 1, hspace=0.3)
    
        for i, fname in enumerate(files):
            path = os.path.join(self.tci_dir, fname)
            ds = xr.open_dataset(path)
            if 'TCI' not in ds:
                continue
    
            da = ds['TCI'].squeeze()
    
            ax = fig.add_subplot(gs[i], projection=ccrs.Robinson() if projection == "Robinson" else ccrs.PlateCarree())
            ax.set_global()

            p = da.plot.pcolormesh(
                ax=ax,
                transform=ccrs.PlateCarree(),
                cmap=self.cmap,
                vmin=self.vmin,
                vmax=self.vmax,
                add_colorbar=False,
                add_labels=False
            )
            ax.coastlines()
            ax.set_title(fname.replace("TCI_obs_window_", "").replace(".nc", ""), fontsize=10)
    
        # Add single shared colorbar
        cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.02])
        cbar = fig.colorbar(p, cax=cbar_ax, orientation='horizontal')
        cbar.set_label("TCI (unitless)")
    
        save_path = os.path.join(self.fig_dir, "TCI_obs_window_grid.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[SAVED] {save_path}")
        plt.close(fig)

    def plot_tci_map(self, filename, title=None, save=True):
        filepath = os.path.join(self.tci_dir, filename)
        if not os.path.exists(filepath):
            print(f"[WARN] Missing TCI file: {filepath}")
            return

        ds = xr.open_dataset(filepath)
        if 'TCI' not in ds:
            raise KeyError(f"[ERROR] 'TCI' variable not found in {filepath}")

        da = ds['TCI'].squeeze()

        fig = plt.figure(figsize=(10, 5))
        ax = plt.axes(projection=ccrs.Robinson())
        ax.set_global()
        im = da.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=self.cmap,
            vmin=self.vmin,
            vmax=self.vmax,
            add_colorbar=False,
            add_labels=False
        )
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
        cbar.set_label('TCI (unitless)')

        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.set_title(title or filename.replace('.nc', ''), fontsize=12)

        if save:
            save_name = filename.replace('.nc', '.png')
            save_path = os.path.join(self.fig_dir, save_name)
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"[SAVED] {save_path}")

        plt.close(fig)

    def plot_all_maps(self):
        for fname in sorted(os.listdir(self.tci_dir)):
            if fname.startswith("TCI_obs_window_") and fname.endswith(".nc"):
                window = fname.split("_window_")[-1].replace(".nc", "")
                self.plot_tci_map(fname, title=f"TCI (Obs) – {window}")

    # ========== REGIONAL MEAN LOADER ==========

    def load_regional_means_with_error(self, directory, prefix="TCI_obs_regional_mean_"):
        files = sorted([
            f for f in os.listdir(directory)
            if f.startswith(prefix) and f.endswith(".nc")
        ])
        region_data = {}

        for fname in files:
            tag = fname.replace(prefix, "").replace(".nc", "")
            time_str = self._extract_window_time(tag)
            ds = xr.open_dataset(os.path.join(directory, fname))

            for var in ds.data_vars:
                base = var.replace("TCI_", "").replace("_stderr", "")
                key = f"{base}_stderr" if "stderr" in var else base

                if base not in region_data:
                    region_data[base] = []

                region_data[base].append({
                    "time": time_str,
                    "value" if "stderr" not in key else "stderr": ds[var].values.item()
                })

        df_dict = {}
        for region, records in region_data.items():
            df = pd.DataFrame(records).groupby("time", as_index=False).first()
            df = df.sort_values("time")
            df_dict[region] = df

        return df_dict

    # ========== REGIONAL TIME SERIES ==========

    def plot_regional_time_series(
        self,
        obs_df_dict,
        model_df_dicts=None,
        model_labels=None,
        show_errorbars=True,
        use_datetime_x=True,
        title="TCI Regional Means: Observations vs Models",
        filename="TCI_obs_vs_model_regional_means.png"
    ):
        fig, ax = plt.subplots(figsize=(11, 6))

        # Plot OBSERVATIONS
        for region, df in obs_df_dict.items():
            x = pd.to_datetime(df["time"]) if use_datetime_x else df["time"]
            y = df["value"]
            yerr = df["stderr"] if show_errorbars and "stderr" in df else None
            ax.errorbar(x, y, yerr=yerr, label=f"{region} (OBS)", fmt='-o')

        # Plot MODEL overlays
        if model_df_dicts:
            for model_label, model_dict in zip(model_labels, model_df_dicts):
                for region, df in model_dict.items():
                    x = pd.to_datetime(df["time"]) if use_datetime_x else df["time"]
                    y = df["value"]
                    yerr = df["stderr"] if show_errorbars and "stderr" in df else None
                    ax.errorbar(
                        x, y, yerr=yerr, linestyle='--', marker='x',
                        label=f"{region} ({model_label})"
                    )

        ax.set_title(title)
        ax.set_ylabel("TCI (unitless)")
        ax.set_xlabel("Time")
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(loc='best', fontsize=9)

        if use_datetime_x:
            ax.xaxis.set_major_formatter(DateFormatter('%Y-%m'))

        plt.xticks(rotation=45)
        plt.tight_layout()

        save_path = os.path.join(self.fig_dir, filename)
        plt.savefig(save_path, dpi=150)
        print(f"[SAVED] {save_path}")
        plt.close(fig)

    # ========== FULL PIPELINE ==========

    def run_all_plots(
        self,
        model_dirs_labels=None,  # List of tuples: (model_dir, label)
        show_errorbars=True
    ):
        print("[INFO] Plotting all maps...")
        self.plot_all_maps()

        print("[INFO] Loading observation regional means...")
        obs_df = self.load_regional_means_with_error(self.regional_dir)

        model_df_list = []
        model_labels = []
        if model_dirs_labels:
            for model_dir, label in model_dirs_labels:
                model_df = self.load_regional_means_with_error(model_dir, prefix="TCI_model_regional_mean_")
                model_df_list.append(model_df)
                model_labels.append(label)

        print("[INFO] Plotting regional time series...")
        self.plot_regional_time_series(
            obs_df_dict=obs_df,
            model_df_dicts=model_df_list,
            model_labels=model_labels,
            show_errorbars=show_errorbars,
            use_datetime_x=True,
        )


In [ ]:
# =============================================================================
# USER SETUP
# Choose: tci_difference, tcc_difference, or observed_tci
# =============================================================================
PRODUCT = "tci_difference"
DATA_ROOT = INPUT_DATA_ROOT
OUTPUT_ROOT = WORKFLOW_DATA_DIR / "dirmeyer_tci"
COUPLING_FIGURE_DIR = WORKFLOW_FIGURE_DIR / "dirmeyer_tci"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COUPLING_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = COUPLING_FIGURE_DIR
LANDMASK_FILE = DATA_ROOT / "reference/lnd_sea_mask/landmask_1x1.nc"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

WINDOWS = {"D1-15": "D1-15", "D16-30": "D16-30", "D31-45": "D31-45", "D46-60": "D46-60"}
REGION = {"lat": (-90, 90), "lon": (-180, 180)}
REFERENCE_EXPERIMENT = "CAPTEN10"
DART_EXPERIMENTS = ("DARTEN20", "DARTEN40")
KEY = "H2OSOI_5cm_LHFLX"


In [ ]:
ensemble_sizes = {"CTRLEN10": 10, "CAPTEN10": 10, "DARTEN20": 20, "DARTEN40": 40}

if PRODUCT == "tci_difference":
    files = {
        experiment: {
            "path": str(OUTPUT_ROOT / "%(exp)s"),
            "nens": nens,
            "template": f"TCI_window_%(window)s_EN%(ensemble)s_{KEY}.nc",
        }
        for experiment, nens in ensemble_sizes.items()
    }
    plotter = TCIPlotter(files, WINDOWS, str(LANDMASK_FILE), compute_ensemble_mean=True)
    plotter.plot_dart_minus_reference_comparison(
        reference_exp=REFERENCE_EXPERIMENT,
        dart_exps=DART_EXPERIMENTS,
        percentile=0.9,
        region=REGION,
        significance=True,
        levels=[-10, -8, -6, -4, -2, -1, 0, 1, 2, 4, 6, 8, 10],
        savepath=str(FIGURE_DIR / f"TCI_{KEY}_p90_dart_minus_reference.pdf"),
    )
elif PRODUCT == "tcc_difference":
    files = {
        experiment: {
            "path": str(OUTPUT_ROOT / "%(exp)s"),
            "nens": nens,
            "template": f"fcst_tcc_rmse_%(exp)s_EN%(ensemble)s_{KEY}_%(window)s.nc",
        }
        for experiment, nens in ensemble_sizes.items()
    }
    plotter = TCCPlotter(
        tci_files_dict=files,
        variable="TCC",
        windows=WINDOWS,
        region=REGION,
        landmask_file=str(LANDMASK_FILE),
        compute_ensemble_mean=True,
    )
    plotter.plot_dart_minus_reference_comparison(
        reference_exp=REFERENCE_EXPERIMENT,
        dart_exps=DART_EXPERIMENTS,
        percentile=0.9,
        levels=[-0.5, -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.5],
        significance=True,
        savepath=str(FIGURE_DIR / f"TCC_{KEY}_p90_dart_minus_reference.pdf"),
    )
elif PRODUCT == "observed_tci":
    obs_dir = OUTPUT_ROOT / "obs_tci"
    plotter = ObservedTCIPlotter(
        tci_dir=str(obs_dir),
        regional_dir=str(obs_dir),
        fig_dir=str(FIGURE_DIR),
        cmap="magma",
        vmin=-5,
        vmax=5,
    )
    plotter.run_all_plots(model_dirs_labels=[])
else:
    raise ValueError(f"Unknown PRODUCT={PRODUCT!r}")
